In [2]:
# Setup the Jupyter version of Dash so the dashboard can run inside Jupyter Lab.
from jupyter_dash import JupyterDash

# Import the dashboard components used for layout, tables, callbacks, charts,
# and the Leaflet-based map.
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64
import os

# This helps JupyterDash correctly route the app inside the notebook environment.
JupyterDash.infer_jupyter_proxy_config()

# Pandas is used to turn MongoDB query results into DataFrames that Dash can
# easily display and manipulate.
import pandas as pd

# Import the AnimalShelter CRUD class
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################
# The project requires use of the AAC MongoDB account. These credentials are
# hardcoded here so the dashboard can connect directly to the database.
username = "aacuser"
password = "password123"   

# Create the CRUD object that connects the dashboard to MongoDB.
db = AnimalShelter(username, password)

# Retrieve all records from the AAC collection so the dashboard begins in an
# unfiltered default state.
df = pd.DataFrame.from_records(db.read({}))

# The CRUD module should already exclude _id, but this extra check prevents
# display issues if _id is ever returned in a future version.
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


#########################
# Filter Query Function
#########################
# This function returns the correct MongoDB query for each rescue category.
# The breed filters use regex so the dashboard can match both exact breed names
# and common mixed-breed variants found in the AAC dataset.
def get_filter_query(filter_type):
    # Water Rescue dogs:
    # Intact females, between 26 and 156 weeks old.
    if filter_type == "Water Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    # Mountain or Wilderness Rescue dogs:
    # Intact males, between 26 and 156 weeks old.
    elif filter_type == "Mountain or Wilderness Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    # Disaster or Individual Tracking dogs:
    # Intact males, between 20 and 300 weeks old.
    elif filter_type == "Disaster or Individual Tracking":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    # Reset / default shows all records
    return {}


#########################
# Dashboard Layout / View
#########################
# Create the Dash application.
app = JupyterDash(__name__)

# Add the Grazioso Salvare logo.
# Place the logo image file in the same Codio folder as the notebook.
# Update the filename if your image uses a different name.
image_filename = "Grazioso Salvare Logo.png"

# Use a fallback name if the primary filename does not exist.
if not os.path.exists(image_filename):
    image_filename = "my-image.png"

# Read and encode the image so it can be displayed directly in the dashboard.
encoded_image = base64.b64encode(open(image_filename, "rb").read())

# Build the dashboard layout.
app.layout = html.Div([
    # Display the company logo as a clickable link and include the unique
    # student identifier required by the project.
    html.Center([
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image.decode()),
                style={"height": "140px"}
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),
        html.H1("Grazioso Salvare Dashboard"),
        html.H3("Rachael - SNHU CS-340 Project Two")
    ]),

    html.Hr(),

    # Create the interactive filter controls the user will use to switch
    # between rescue categories or return to the full dashboard view.
    html.Div([
        html.Label("Rescue Category Filter:"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": " Reset", "value": "Reset"},
                {"label": " Water Rescue", "value": "Water Rescue"},
                {"label": " Mountain Rescue", "value": "Mountain or Wilderness Rescue"},
                {"label": " Disaster Rescue", "value": "Disaster or Individual Tracking"}
            ],
            value="Reset",
            inline=True
        )
    ]),

    html.Hr(),

    # Create the interactive data table that displays AAC records from MongoDB.
    # The table is configured to allow only one selected row so the map can
    # always use one clear record for the marker display.
    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict("records"),

        # Single-row selection is required for the geolocation map callback.
        row_selectable="single",

        # Select the first row by default so the map and charts load immediately.
        selected_rows=[0],

        # These features improve usability by allowing the client to sort,
        # filter, and page through the data more easily.
        sort_action="native",
        filter_action="native",
        page_action="native",
        page_current=0,
        page_size=10,

        # Style settings improve readability for a wide data table.
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "minWidth": "120px",
            "width": "120px",
            "maxWidth": "180px",
            "whiteSpace": "normal"
        },
        style_header={
            "backgroundColor": "rgb(230, 230, 230)",
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),

    # Place the chart and the map side by side so the dashboard presents both
    # visualizations together on the same screen.
    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(id="graph-id", className="col s12 m6"),
            html.Div(id="map-id", className="col s12 m6")
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

# This callback updates the data table whenever the user selects a rescue filter.
# It runs the correct MongoDB query through the CRUD module and returns the new
# table data to the dashboard.
@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    # Get the correct MongoDB query for the chosen rescue category.
    query = get_filter_query(filter_type)

    # Pull the filtered records from MongoDB and convert them to a DataFrame.
    dff = pd.DataFrame.from_records(db.read(query))

    # Remove _id if it appears so the table stays clean and readable.
    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    # Return the filtered records to the Dash data table.
    return dff.to_dict("records")


# This callback creates the second dashboard chart from the records currently
# visible in the data table. A pie chart is used here to match the project
# examples more closely.
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    # If no filtered view exists yet, use the full starting DataFrame.
    if viewData is None or len(viewData) == 0:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # If the filtered results are empty, show a helpful message instead of crashing.
    if dff.empty:
        return [html.H4("No data to display")]

    # Count breeds in the currently visible records.
    breed_counts = dff["breed"].value_counts().reset_index()
    breed_counts.columns = ["breed", "count"]

    # Limit the chart to the top 10 breeds so the pie chart stays readable.
    breed_counts = breed_counts.head(10)

    # Build the breed distribution pie chart.
    fig = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Breed Distribution"
    )

    # Return the chart to the dashboard.
    return [dcc.Graph(figure=fig)]


# This callback highlights selected columns in the table. Returning an empty
# list prevents errors when no columns are selected.
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


# This callback updates the geolocation chart based on the currently visible
# table data and the selected row. The selected row determines which animal's
# latitude and longitude are displayed on the map.
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    # Use the currently visible table data so the map stays synced with filtering.
    if viewData is None or len(viewData) == 0:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # If no valid data is available, show a message instead of crashing.
    if dff.empty:
        return [html.H4("No map data available")]

    # Because only one row can be selected, use the first row by default when
    # nothing is actively selected.
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # If filtering reduced the number of rows, reset to row 0 to avoid an error.
    if row >= len(dff):
        row = 0

    # Create the map centered on Austin, Texas, and place a marker at the
    # latitude and longitude of the selected animal record.
    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row]["location_lat"], dff.iloc[row]["location_long"]],
                    children=[
                        # Show breed in the tooltip and the animal name in the popup.
                        dl.Tooltip(str(dff.iloc[row]["breed"])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(dff.iloc[row]["name"]))
                        ])
                    ]
                )
            ]
        )
    ]


# Run the dashboard in JupyterLab.
app.run_server()

Dash app running on https://nightsimon-atlassafari-3000.codio.io/proxy/8050/
